# Getting Started
* Create a new API key at https://www.console.anthropic.com
* Create a .env file
  * Add ```ANTHROPIC_API_KEY="<your_api_key>"```
* Ensure uv is installed: https://docs.astral.sh/uv/getting-started/installation/
* Run Standalone Notebook:
  * > uv run jupyter lab
* Run Notebook in VSCode:
  * Open Jupyter Notebook and select `.venv (Python 3.12.11)` as the kernel

In [5]:
# environment setup
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import ToolParam, Message
import json
client = Anthropic()
model = 'claude-haiku-4-5'
max_tokens = 1000

In [6]:
# helper functions
def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)

def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None, temperature=1.0, tools=[]):
  params = {
    'model': model,
    'max_tokens': max_tokens,
    'messages': messages,
    'temperature': temperature,
  }

  if tools:
    params['tools'] = tools

  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)

  return message

def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[]
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)

def text_from_message(message):
  return "\n".join([
    block.text for block in message.content if block.type == "text"
  ])

# Web Search Tool
Claude includes built in web search tool that lets it search the internet for current or specialized information to answer user questinos. Unlike other tools where you have to provide the function implementation, Claude handles the entire search process automatically. You just need the schema to enable it

**Note**: You must enable web search in the settings console before using it: https://console.anthropic.com/settings/privacy

In [ ]:
# Include web schema
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search", 
    "max_uses": 5 # limits how many searches Claude can perform. Claude might do follow up searches so this can limit it
    # "allowed_domains" = ["marketwatch.com"] # allow only specific domains
}

In [10]:
messages = []
add_user_message(
    messages,
    "What happened in the market today",
)

response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_011Cd4oPQfTKVsLHDb3Bj1LV', container=None, content=[TextBlock(citations=None, text="I'll search for today's market information for you.", type='text'), ServerToolUseBlock(id='srvtoolu_01Y7CxUBGThHTNPfo7am4wZe', caller=None, input={'query': 'market today July 16 2026'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EpAgCioIERgCIiQ0Nzc2NGYzMS1lZWY2LTRmM2EtOTRlMS0zZDZkZDFmMTY2ODYSDDpv7hODr98QG0v9NRoM20ilx9ZeVJ7RxHNUIjDa1evRxq/5CbHUskGT7a7ILbQFAPnkKCu4GPDO8WH3w2ONu4dKgInm12vjspR52Xkqkx+vKxYwpSC903W2ceP8e2zfcBei8qVytf6/VgeQ1kUeaRrjhr36cqxellI5M5ynEuVzNJO2OZi8XPhxfYvnBWTW4Zn0JqLo0dEXyEGaraQ4PQEroUOnmxKEqiOxieKua/OiJPIrvhLzNqiTvQ/K29OjaKiUboezNFp+j5F1tSEpOHQvxGMaSAhDM69WqJEaTfpcSC9JREIGUQUvuwsdJ2rXFti3In8i718uSnJf+/R3JzaTkfbp8lhB1JjSmAl4Q0Wz6bwzSPAQ8neHkWfPCkzfMfFXnBtJZ2F7ujH2YT5ojNiO725U4Ocju2ijkIypxYa8+V2uJvof9w26IHo3Ugt8FCg5mSUGec9DPtMBjJXrZe6iE5ODaoHnKjwY1J0IvZSStVgr

### Web Search Response
Response contains several types of blocks you can render back to user
* **Text Block** - Claude's explanation of what it's doing 
* **ServerToolUseBlock** - Shows the exact search query Claude used
* **WebSearchToolResultBlock** - Contains the search results
* **WebSearchResultBlock** - Individual search results within titles and URLs
* **Citation blocks** - Text that supports Claude's statements